# Project 3 — Early Warning System for Sepsis (PhysioNet 2019)
## Notebook 01 — Exploration, Cohort Definition, and Target Construction

**Goal of this notebook**
- Understand the dataset structure and cohort (patient-level time series).
- Define the prediction task: *predict sepsis onset within the next* **H** *hours*.
- Construct a **leakage-safe** target `y_within_H`.
- Save clean artifacts for downstream notebooks (02b+).

**Key rule**
- All modeling/evaluation later will use **patient-grouped** splits (`patient_id` groups).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)

# -------------------------
# Paths (EDIT THIS ONCE)
# -------------------------
# Recommended: keep a repo/project folder like:
#   project_root/
#     training/training_setA/*.psv
#     training/training_setB/*.psv
#     results/
#     figures/
#
# If you're running inside that folder, Path('.') works.
PROJECT_ROOT = Path(".").resolve()

# If your dataset lives elsewhere, set DATA_ROOT explicitly, e.g.:
# DATA_ROOT = Path(r"F:\Project 3 Folder\project 3--ews")
DATA_ROOT = PROJECT_ROOT

TRAIN_A = DATA_ROOT / "training" / "training_setA"
TRAIN_B = DATA_ROOT / "training" / "training_setB"

RESULTS_DIR = DATA_ROOT / "results"
FIG_DIR = DATA_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_A exists:", TRAIN_A.exists(), TRAIN_A)
print("TRAIN_B exists:", TRAIN_B.exists(), TRAIN_B)


### 1) Load a patient sample (fast iteration)
We start with a reproducible subset of patients so we can iterate quickly.  
Later, set `N_PATIENTS=None` to load the full dataset.


In [ ]:
# -------------------------
# Sampling parameters
# -------------------------
RANDOM_SEED = 42
N_PATIENTS = 500   # set to None to load ALL patients

files_a = sorted(TRAIN_A.glob("*.psv"))
files_b = sorted(TRAIN_B.glob("*.psv"))
all_files = files_a + files_b

print("Total patient files found:", len(all_files))

rng = np.random.default_rng(RANDOM_SEED)
if N_PATIENTS is None:
    sample_files = all_files
else:
    sample_files = rng.choice(all_files, size=min(N_PATIENTS, len(all_files)), replace=False)
    sample_files = sorted([Path(f) for f in sample_files])

print("Using patient files:", len(sample_files))
print("Example file:", sample_files[0].name if len(sample_files) else None)


### 2) Load and concatenate patient time-series
- Each file is one patient
- Rows are hourly measurements in ICU
- We use `ICULOS` as the hour index (should be present in this dataset)


In [ ]:
def load_patient_psv(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp, sep="|")
    df["patient_id"] = fp.stem

    # Ensure ICULOS exists and is numeric
    if "ICULOS" not in df.columns:
        # Fallback (shouldn't happen in PhysioNet 2019)
        df["ICULOS"] = np.arange(1, len(df) + 1, dtype=float)
    df["ICULOS"] = pd.to_numeric(df["ICULOS"], errors="coerce")

    # Ensure SepsisLabel exists
    if "SepsisLabel" in df.columns:
        df["SepsisLabel"] = pd.to_numeric(df["SepsisLabel"], errors="coerce").fillna(0).astype(int)

    return df

dfs = [load_patient_psv(fp) for fp in sample_files]
data = pd.concat(dfs, ignore_index=True)

data = data.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

print("Rows:", data.shape[0], "Cols:", data.shape[1])
print("Patients:", data["patient_id"].nunique())
data.head()


### 3) Cohort summary (patient length-of-stay in hours)
We report **patient-level** length statistics (rows per patient).


In [ ]:
steps_per_patient = data.groupby("patient_id").size()

cohort_summary = pd.DataFrame({
    "n_patients": [data["patient_id"].nunique()],
    "n_rows": [len(data)],
    "median_hours": [float(steps_per_patient.median())],
    "p90_hours": [float(steps_per_patient.quantile(0.90))],
    "p95_hours": [float(steps_per_patient.quantile(0.95))],
    "max_hours": [int(steps_per_patient.max())],
})

cohort_summary.to_csv(RESULTS_DIR / "cohort_summary.csv", index=False)
cohort_summary


In [ ]:
plt.figure()
steps_per_patient.hist(bins=40)
plt.title("ICU length-of-stay (rows per patient) — sample")
plt.xlabel("Hours (rows)")
plt.ylabel("Patients")
plt.tight_layout()
plt.savefig(FIG_DIR / "los_hist_sample.png", dpi=200)
plt.show()


### 4) Define sepsis onset time (patient-level)
We define `event_iculos` as the **first hour where `SepsisLabel == 1`** for that patient.

This becomes our reference “onset” time for:
- prevalence
- lead-time evaluation later
- building the “within H hours” label


In [ ]:
assert "SepsisLabel" in data.columns, "SepsisLabel not found. Check file format."

sepsis_any = data.groupby("patient_id")["SepsisLabel"].max().astype(int)

def first_event_hour(patient_df: pd.DataFrame) -> float:
    rows = patient_df[patient_df["SepsisLabel"] == 1]
    if rows.empty:
        return np.nan
    return float(rows["ICULOS"].iloc[0])

event_time = (
    data.sort_values(["patient_id", "ICULOS"])
        .groupby("patient_id", as_index=False)
        .apply(lambda g: pd.Series({"event_iculos": first_event_hour(g)}))
)

patient_events = (
    event_time.merge(sepsis_any.rename("sepsis_any"), left_on="patient_id", right_index=True, how="left")
)

# Add LOS for convenience
patient_events = patient_events.merge(
    steps_per_patient.rename("los_hours"), left_on="patient_id", right_index=True, how="left"
)

patient_events.head()


In [ ]:
label_summary = pd.DataFrame({
    "n_patients": [int(patient_events.shape[0])],
    "n_sepsis_patients": [int(patient_events["sepsis_any"].sum())],
    "n_nonsepsis_patients": [int((1 - patient_events["sepsis_any"]).sum())],
    "sepsis_prevalence_patient_level": [float(patient_events["sepsis_any"].mean())],
    "median_event_hour": [float(np.nanmedian(patient_events["event_iculos"]))],
})

label_summary.to_csv(RESULTS_DIR / "label_timing_summary.csv", index=False)
label_summary


In [ ]:
plt.figure()
patient_events["event_iculos"].dropna().hist(bins=30)
plt.title("First sepsis label time (ICULOS) — sample")
plt.xlabel("ICULOS (hour)")
plt.ylabel("Sepsis patients")
plt.tight_layout()
plt.savefig(FIG_DIR / "onset_hist_sample.png", dpi=200)
plt.show()


### 5) Construct the target: `y_within_H`
We want an **early warning** label:

For each patient at hour `t`,  
`y_within_H(t) = 1` if sepsis onset happens in **(t, t+H]**.

So:
- if onset is 10 and H=6
- then hours t=4..9 (exclusive of 10?):
  - for t=4: 4 < 10 <= 10 ✅ → positive
  - for t=10: 10 < 10 ❌ → not positive

**Important:** rows at/after onset should NOT be used for training an early warning model,  
otherwise you can get paradoxical labels (post-onset physiology but label=0).
So we will **censor** rows with `ICULOS >= event_iculos` for sepsis patients.


In [ ]:
H = 6  # prediction horizon in hours

# Attach event time per row
event_dict = dict(zip(patient_events["patient_id"], patient_events["event_iculos"]))
data["event_iculos"] = data["patient_id"].map(event_dict)

# Create y_within_H
t = data["ICULOS"].values
t_event = data["event_iculos"].values

data["y_within_H"] = 0
mask_has_event = ~np.isnan(t_event)
data.loc[mask_has_event, "y_within_H"] = ((t[mask_has_event] < t_event[mask_has_event]) &
                                          (t_event[mask_has_event] <= (t[mask_has_event] + H))).astype(int)

# Censor post-onset rows for sepsis patients (ICULOS >= event_iculos)
data["use_row"] = True
data.loc[mask_has_event & (data["ICULOS"] >= data["event_iculos"]), "use_row"] = False

print("Rows before censoring:", len(data))
print("Rows after censoring:", int(data["use_row"].sum()))

# Row-level prevalence (on training rows only)
row_prev = data.loc[data["use_row"], "y_within_H"].mean()
print("Row-level prevalence of y_within_H (training rows):", float(row_prev))

data.loc[data["use_row"], ["patient_id", "ICULOS", "SepsisLabel", "event_iculos", "y_within_H"]].head(20)


### 6) Missingness analysis (overall + by sepsis vs non-sepsis)
Missingness patterns themselves are predictive in ICU datasets, so we quantify them explicitly.


In [ ]:
# Focus on rows used for training (pre-onset / all non-sepsis)
train_rows = data.loc[data["use_row"]].copy()

feature_cols = [c for c in train_rows.columns if c not in ["patient_id", "ICULOS", "SepsisLabel", "event_iculos", "y_within_H", "use_row"]]

missing_overall = train_rows[feature_cols].isna().mean().sort_values(ascending=False)
missing_report = missing_overall.reset_index()
missing_report.columns = ["feature", "missing_fraction"]
missing_report.to_csv(RESULTS_DIR / "missingness_report.csv", index=False)

missing_report.head(20)


In [ ]:
# Missingness by sepsis_any (patient label)
train_rows = train_rows.merge(patient_events[["patient_id", "sepsis_any"]], on="patient_id", how="left")

missing_by_group = (
    train_rows.groupby("sepsis_any")[feature_cols]
        .apply(lambda g: g.isna().mean())
        .T
        .rename(columns={0: "missing_nonsepsis", 1: "missing_sepsis"})
)

missing_by_group["diff_sepsis_minus_nonsepsis"] = missing_by_group["missing_sepsis"] - missing_by_group["missing_nonsepsis"]
missing_by_group = missing_by_group.sort_values("diff_sepsis_minus_nonsepsis", ascending=False)

missing_by_group.head(20)


### 7) Quick vitals sanity plots (optional but useful)
We plot a few core vitals to sanity-check ranges and distributions.


In [ ]:
vitals = [c for c in ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"] if c in train_rows.columns]
print("Vitals present:", vitals)

for v in vitals[:6]:
    plt.figure()
    train_rows[v].dropna().hist(bins=50)
    plt.title(f"{v} distribution (pre-onset / non-sepsis rows)")
    plt.xlabel(v)
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"dist_{v}.png", dpi=200)
    plt.show()


### 8) Save artifacts for downstream notebooks
We save:
- `patient_events.parquet`: patient-level onset & labels
- `sample_long_table.parquet`: long table with target + censoring indicator

Notebook 02b will load `sample_long_table.parquet` and engineer leakage-safe temporal features.


In [ ]:
# Patient-level artifact
patient_events.to_parquet(RESULTS_DIR / "patient_events.parquet", index=False)

# Row-level artifact (training rows only, plus some columns for later)
out_long = train_rows.drop(columns=["sepsis_any"])  # keep row-level table clean
out_long.to_parquet(RESULTS_DIR / "sample_long_table.parquet", index=False)

print("Saved:")
print(" -", RESULTS_DIR / "patient_events.parquet")
print(" -", RESULTS_DIR / "sample_long_table.parquet")
